# Lab 4. Naive Bayes

## I. Phân loại Naive Bayes sử dụng thư viện sklearn trên dataset nhỏ 

### Bài toán dự đoán rủi ro tín dụng
- Cho danh sách những người vay tiền với các đặc trưng được quan sát bao gồm: Người vay có sở hữu nhà ở hay không (Home Owner); Tình trạng hôn nhân (Marital Status); Thu nhập hàng năm (Annual Income); và Nhận định người vay có bị vỡ nợ hay không (Defaulted Borrower). Trong đó Defaulted Borrower là nhãn lớp cho biết người vay có trả được khoản tiền đã vay hay không? Dữ liệu được lưu trong file credit_data.txt.
- Yêu cầu: Sử dụng thuật toán Naïve Bayes để dự đoán xác suất một người có vỡ nợ hay không dựa vào các đặc trưng Home Owner, Marital Status và Annual Income.

### 1. Import các thư viện cần thiết

In [6]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB, ComplementNB

### 2. Cấu hình dữ liệu

In [7]:
data = pd.DataFrame([
    ["Yes", "Single", "High", "No"],
    ["No", "Married", "High", "No"],
    ["No", "Single", "Low", "No"],
    ["Yes", "Married", "High", "No"],
    ["No", "Divorced", "Low", "Yes"],
    ["No", "Married", "Low", "No"],
    ["Yes", "Divorced", "High", "No"],
    ["No", "Single", "Low", "Yes"],
    ["No", "Married", "Low", "No"],
    ["No", "Single", "Low", "Yes"]
], columns=["Home Owner", "Marital Status", "Annual Income", "Defaulted Borrower"])

In [8]:
# Yêu cầu 1: Chuẩn bị dữ liệu cho mô hình học máy (X: lưu giá trị của các cột đặc trưng; y: lưu giá trị cột nhãn)
######
# Code
X = data[["Home Owner", "Marital Status", "Annual Income"]]
y = data["Defaulted Borrower"]
print("X shape", X.shape)
print(X)
print("y shape", y.shape)
print("y values", y.values)

######

X shape (10, 3)
  Home Owner Marital Status Annual Income
0        Yes         Single          High
1         No        Married          High
2         No         Single           Low
3        Yes        Married          High
4         No       Divorced           Low
5         No        Married           Low
6        Yes       Divorced          High
7         No         Single           Low
8         No        Married           Low
9         No         Single           Low
y shape (10,)
y values ['No' 'No' 'No' 'No' 'Yes' 'No' 'No' 'Yes' 'No' 'Yes']


In [9]:
# Yêu cầu 2: Chuẩn hóa dữ liệu từ dạng phân loại sang dạng số học.
# Gợi ý: Có thể sử dụng các kỹ thuật Label Encoding, One-Hot Encoding,...
######
# Code

# label encoding cho 2 đặc trưng
le_home = LabelEncoder()
X["Home Owner"] = le_home.fit_transform(X["Home Owner"])
le_income = LabelEncoder()
X["Annual Income"] = le_income.fit_transform(X["Annual Income"])
# one-hote encoding cho Marital income
X = pd.get_dummies(X, columns=["Marital Status"], prefix="Marital",dtype=int )
# label encoding cho nhãn
le_targets = LabelEncoder()
y_encoded = le_targets.fit_transform(y)
######

C:\Users\Administrator\AppData\Local\Temp\ipykernel_20052\1164901914.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Home Owner"] = le_home.fit_transform(X["Home Owner"])
C:\Users\Administrator\AppData\Local\Temp\ipykernel_20052\1164901914.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Annual Income"] = le_income.fit_transform(X["Annual Income"])


In [10]:
X

,Home Owner,Annual Income,Marital_Divorced,Marital_Married,Marital_Single
0,1,0,0,0,1
1,0,0,0,1,0
2,0,1,0,0,1
3,1,0,0,1,0
4,0,1,1,0,0
5,0,1,0,1,0
6,1,0,1,0,0
7,0,1,0,0,1
8,0,1,0,1,0
9,0,1,0,0,1


### 3. Training và Testing mô hình Naive Bayes với các phân phối xác suất khác nhau

In [11]:
# Yêu cầu 3: Chuẩn bị một mẫu dữ liệu mới và chuẩn hóa về giá trị số để kiểm tra kết quả phân lớp của mô hình
######
# Code
new_sample = pd.DataFrame([['No', 'Married', 'High']], columns=['Home Owner', 'Marital Status', 'Annual Income'])

new_sample["Home Owner"] = le_home.fit_transform(new_sample["Home Owner"])
new_sample["Annual Income"] = le_income.fit_transform(new_sample["Annual Income"])
# one-hote encoding cho Marital income
new_sample = pd.get_dummies(new_sample, columns=["Marital Status"], prefix="Marital")
######

In [12]:
# 4.4 Đảm bảo mẫu mới có đúng các cột như dữ liệu huấn luyện
for col in X.columns:
    if col not in new_sample.columns:
        new_sample[col] = 0 # Thêm cột còn thiếu với giá trị 0
# 4.5 Sắp xếp cột theo đúng thứ tự của X_train
new_sample = new_sample[X.columns]
new_sample

,Home Owner,Annual Income,Marital_Divorced,Marital_Married,Marital_Single
0,0,0,0,True,0


In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test,y_train, y_test = train_test_split(X, y_encoded, test_size=0.3,stratify=y_encoded, random_state=42)

In [19]:
# Yêu cầu 4: Huấn luyện và kiểm tra mô hình Gaussian Naive Bayes (sử dụng phân phối chuẩn Gauss)
######
# Code
gnb = GaussianNB()
gnb.fit(X_train,y_train)
#dự đoán
y_predict = gnb.predict(X_test)

######

In [20]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print(f"độ chính xác accuracy: {accuracy_score(y_pred=y_predict, y_true=y_test)}")
print(f"Ma trận nhầm lẫn {confusion_matrix(y_true=y_test, y_pred=y_predict)}")
print("\nBáo cáo phân loại:")
print(classification_report(y_test, y_predict, target_names=le_targets.classes_, zero_division=0))

độ chính xác accuracy: 0.6666666666666666
Ma trận nhầm lẫn [[2 0]
 [1 0]]

Báo cáo phân loại:
              precision    recall  f1-score   support

          No       0.67      1.00      0.80         2
         Yes       0.00      0.00      0.00         1

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3



In [21]:
# Yêu cầu 5: Huấn luyện và kiểm tra mô hình Multinomial Naive Bayes (sử dụng phân phối đa thức)
######
# Code
X_train, X_test,y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, stratify=y_encoded,random_state=42)
mnb =MultinomialNB()
mnb.fit(X_train,y_train)
y_predict = mnb.predict(X_test)
print(f"Độ chính xác accuracy: {accuracy_score(y_true=y_test, y_pred=y_predict)}")
print(f"Ma trận nhầm lẫn confusion matric {confusion_matrix(y_true=y_test, y_pred=y_predict)}")
print("\nBáo cáo phân loại (Classification Report):")
print(classification_report(y_test, y_predict,
target_names=le_targets.classes_,
zero_division=0))

######

Độ chính xác accuracy: 0.6666666666666666
Ma trận nhầm lẫn confusion matric [[2 0]
 [1 0]]

Báo cáo phân loại (Classification Report):
              precision    recall  f1-score   support

          No       0.67      1.00      0.80         2
         Yes       0.00      0.00      0.00         1

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3



In [22]:
# Yêu cầu 6: Huấn luyện và kiểm tra mô hình mô hình Bernoulli Naive Bayes (sử dụng phân phối Bernoulli)
######
# Code
X_train, X_test,y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, stratify=y_encoded,random_state=42)
bnb =BernoulliNB()
bnb.fit(X_train,y_train)
y_predict = bnb.predict(X_test)
print(f"Độ chính xác accuracy: {accuracy_score(y_true=y_test, y_pred=y_predict)}")
print(f"Ma trận nhầm lẫn confusion matric {confusion_matrix(y_true=y_test, y_pred=y_predict)}")
print("\nBáo cáo phân loại (Classification Report):")
print(classification_report(y_test, y_predict,
target_names=le_targets.classes_,
zero_division=0))

######

Độ chính xác accuracy: 0.6666666666666666
Ma trận nhầm lẫn confusion matric [[2 0]
 [1 0]]

Báo cáo phân loại (Classification Report):
              precision    recall  f1-score   support

          No       0.67      1.00      0.80         2
         Yes       0.00      0.00      0.00         1

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3



In [23]:
# Yêu cầu 7: Huấn luyện và kiểm tra mô hình Categorical Naive Bayes (Categorical distribution - mở rộng của phân phối Bernoulli)
######
# Code

X_train, X_test,y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, stratify=y_encoded,random_state=42)
cnb = CategoricalNB(
alpha=1.0, # Laplace smoothing (tránh xác suất zero)
fit_prior=True, # Học prior từ dữ liệu
class_prior=None # Có thể gán prior thủ công
)
cnb.fit(X_train, y_train, sample_weight=None)
y_predict = cnb.predict(X_test)
print(f"Độ chính xác accuracy: {accuracy_score(y_true=y_test, y_pred=y_predict)}")
print(f"Ma trận nhầm lẫn confusion matric {confusion_matrix(y_true=y_test, y_pred=y_predict)}")
print("\nBáo cáo phân loại (Classification Report):")
print(classification_report(y_test, y_predict,
target_names=le_targets.classes_,
zero_division=0))
######

Độ chính xác accuracy: 0.6666666666666666
Ma trận nhầm lẫn confusion matric [[2 0]
 [1 0]]

Báo cáo phân loại (Classification Report):
              precision    recall  f1-score   support

          No       0.67      1.00      0.80         2
         Yes       0.00      0.00      0.00         1

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3



In [24]:
# Yêu cầu 8: Huấn luyện và kiểm tra mô hình Complement Naive Bayes (điều chỉnh của phân phối đa thức)
######
# Code
X_train, X_test,y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, stratify=y_encoded,random_state=42)
cnb = ComplementNB(
alpha=1.0, # Laplace smoothing
fit_prior=True, # Học prior từ dữ liệu
norm=False # Không chuẩn hóa features
)
cnb.fit(X_train, y_train)
y_predict = cnb.predict(X_test)
print(f"Độ chính xác accuracy: {accuracy_score(y_true=y_test, y_pred=y_predict)}")
print(f"Ma trận nhầm lẫn confusion matric {confusion_matrix(y_true=y_test, y_pred=y_predict)}")
print("\nBáo cáo phân loại (Classification Report):")
print(classification_report(y_test, y_predict,
target_names=le_targets.classes_,
zero_division=0))
######

######

Độ chính xác accuracy: 1.0
Ma trận nhầm lẫn confusion matric [[2 0]
 [0 1]]

Báo cáo phân loại (Classification Report):
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         2
         Yes       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



## II. Phân loại Naive Bayes sử dụng thư viện sklearn trên dataset lớn (Dữ liệu: credit_data.csv)

- Bài toán: Dự đoán điểm tín dụng của khách hàng khi vay vốn sử dụng Naïve Bayes.
- Mục tiêu:
	- Xây dựng được mô hình Naïve Bayes sử dụng thư viện sklearn.
	- Ứng dụng và hiểu cách áp dụng mô hình Naïve Bayes vào giải quyết bài toán thực tế (ví dụ: dự đoán điểm tín dụng).
	- Sử dụng độ đo Accuracy để đánh giá chất lượng mô hình.
	- Thay đổi các phân bố xác suất (phân phối chuẩn, phân phối đa thức, phân phối Bernoulli) để chọn ra bộ phận lớp Naive Bayes phù hợp với dữ liệu.
- Dữ liệu:
	- Tập các đặc trưng của khách hàng và điểm tín dụng tương ứng trong một khoảng thời gian nhất định.
	- Tập các nhãn (Cột “Risk”): Gồm 2 loại nhãn “Good” và “Bad”. Trong đó “Good” biểu thị khách hàng có khả năng trả nợ đúng hạn, “Bad” biểu thị khách hàng có khả năng vỡ nợ.
- Loại bài toán: Phân loại
	- Input: n vector đã mã hóa của các khách hàng.
	- Output: nhãn y là một trong 2 nhãn trên.


### 1. Import các thư  viện cần thiết

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score

### 2. Load dữ liệu

In [26]:
# Yêu cầu 1: Đọc dữ liệu từ file csv, hiển thị 5 dòng đầu tiên trong dataset
######
# Code

path = r'Z:\DOCUMENTS\KPDL\DataMining-Homework\Lab4-Naive Bayes\credit_data.csv'
df = pd.read_csv(path)
data = df.drop("Unnamed: 0",axis=1)
data.head()
######

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,49,male,1,own,little,NaN,2096,12,education,good
3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,53,male,2,free,little,little,4870,24,car,bad


### 3. Tiền xử lý dữ liệu

#### 3.1. Hiểu dữ liệu

In [27]:
# Yêu cầu 2: Hiển thị thông tin tổng quan của dataset (số dòng, số cột, tên các cột, kiểu dữ liệu)
# Viết nhận xét về kết quả thu được
######
# Code
print(data.shape)
print(f"Tên các cột: {data.columns.tolist()}")
data.info()
######
# Nhận xét: Dataset gồm 1000 mẫu và 10 đặc trưng.
# Có cả đặc trưng dạng số (Age, Job, Credit amount, Duration)
# và dạng phân loại (Sex, Housing, Saving accounts, Checking account, Purpose, Risk).
# Một số cột có missing values (Saving accounts, Checking account).
######

######

(1000, 10)
Tên các cột: ['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose', 'Risk']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Age               1000 non-null   int64 
 1   Sex               1000 non-null   object
 2   Job               1000 non-null   int64 
 3   Housing           1000 non-null   object
 4   Saving accounts   817 non-null    object
 5   Checking account  606 non-null    object
 6   Credit amount     1000 non-null   int64 
 7   Duration          1000 non-null   int64 
 8   Purpose           1000 non-null   object
 9   Risk              1000 non-null   object
dtypes: int64(4), object(6)
memory usage: 78.2+ KB


In [28]:
# Yêu cầu 3: Hiển thị thông tin các thống kê mô tả cơ bản của các đặc trưng
# Gợi ý: Sử dụng hàm describe()
######
# Code
print('--- Thống kê mô tả ---')
df.describe(include='all')

######

--- Thống kê mô tả ---


,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
count,1000.000000,1000.000000,1000,1000.000000,1000,817,606,1000.000000,1000.000000,1000,1000
unique,NaN,NaN,2,NaN,3,4,3,NaN,NaN,8,2
top,NaN,NaN,male,NaN,own,little,little,NaN,NaN,car,good
freq,NaN,NaN,690,NaN,713,603,274,NaN,NaN,337,700
mean,499.500000,35.546000,NaN,1.904000,NaN,NaN,NaN,3271.258000,20.903000,NaN,NaN
std,288.819436,11.375469,NaN,0.653614,NaN,NaN,NaN,2822.736876,12.058814,NaN,NaN
min,0.000000,19.000000,NaN,0.000000,NaN,NaN,NaN,250.000000,4.000000,NaN,NaN
25%,249.750000,27.000000,NaN,2.000000,NaN,NaN,NaN,1365.500000,12.000000,NaN,NaN
50%,499.500000,33.000000,NaN,2.000000,NaN,NaN,NaN,2319.500000,18.000000,NaN,NaN
75%,749.250000,42.000000,NaN,2.000000,NaN,NaN,NaN,3972.250000,24.000000,NaN,NaN


#### 3.2. Kiểm tra missing values

In [29]:
# Yêu cầu 4: Thống kê tổng số giá trị bị thiếu trong dataset, liệt kê giá trị bị thiếu trong mỗi cột.
# Viết nhận xét cho kết quả thu được
######
# Code
missing_total = df.isnull().sum().sum()
missing_per_col = df.isnull().sum()
print(f'Tổng số giá trị bị thiếu: {missing_total}')
print('\nSố giá trị bị thiếu theo từng cột:')
print(missing_per_col[missing_per_col > 0])
print('\nTỷ lệ missing (%):')
print((missing_per_col[missing_per_col > 0] / len(df) * 100).round(2))

######

Tổng số giá trị bị thiếu: 577

Số giá trị bị thiếu theo từng cột:
Saving accounts     183
Checking account    394
dtype: int64

Tỷ lệ missing (%):
Saving accounts     18.3
Checking account    39.4
dtype: float64


#### 3.3. Xử lý missing values

In [30]:
# Yêu cầu 5: Lựa chọn và áp dụng phương pháp xử lý các giá trị bị thiếu
######
# Code
df_clean = df.copy()
for col in ['Saving accounts', 'Checking account']:
    mode_val = df_clean[col].mode()[0]
    df_clean[col].fillna(mode_val, inplace=True)
    print(f"Cột '{col}': đã điền missing bằng mode = '{mode_val}'")
print('\nKiểm tra lại missing values:')
print(df_clean.isnull().sum())

######

Cột 'Saving accounts': đã điền missing bằng mode = 'little'
Cột 'Checking account': đã điền missing bằng mode = 'little'

Kiểm tra lại missing values:
Unnamed: 0          0
Age                 0
Sex                 0
Job                 0
Housing             0
Saving accounts     0
Checking account    0
Credit amount       0
Duration            0
Purpose             0
Risk                0
dtype: int64


C:\Users\Administrator\AppData\Local\Temp\ipykernel_20052\3748295626.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(mode_val, inplace=True)


#### 3.4. Mã hóa các đặc trưng rời rạc

In [31]:
# Yêu cầu 6: Chuyển đổi các giá trị rời rạc về giá trị số
# Gợi ý: Có thể áp dụng một trong các kỹ thuật: Label Encoding, One-Hot Encoding, Ordinal Encoding, Target Encoding
######
# Code
from sklearn.preprocessing import LabelEncoder
df_encoded = df_clean.copy()
categorical_cols = df_encoded.select_dtypes(include=['object']).columns.tolist()
print('Các cột phân loại cần mã hóa:', categorical_cols)
label_encoders = {}
for col in categorical_cols:
    le2 = LabelEncoder()
    df_encoded[col] = le2.fit_transform(df_encoded[col])
    label_encoders[col] = le2
    print(f" {col}: {dict(zip(le2.classes_, le2.transform(le2.classes_)))}")
print('\nDataset sau mã hóa (5 dòng đầu):')
print(df_encoded.head())

######

Các cột phân loại cần mã hóa: ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose', 'Risk']
 Sex: {'female': np.int64(0), 'male': np.int64(1)}
 Housing: {'free': np.int64(0), 'own': np.int64(1), 'rent': np.int64(2)}
 Saving accounts: {'little': np.int64(0), 'moderate': np.int64(1), 'quite rich': np.int64(2), 'rich': np.int64(3)}
 Checking account: {'little': np.int64(0), 'moderate': np.int64(1), 'rich': np.int64(2)}
 Purpose: {'business': np.int64(0), 'car': np.int64(1), 'domestic appliances': np.int64(2), 'education': np.int64(3), 'furniture/equipment': np.int64(4), 'radio/TV': np.int64(5), 'repairs': np.int64(6), 'vacation/others': np.int64(7)}
 Risk: {'bad': np.int64(0), 'good': np.int64(1)}

Dataset sau mã hóa (5 dòng đầu):
   Unnamed: 0  Age  Sex  Job  Housing  Saving accounts  Checking account  \
0           0   67    1    2        1                0                 0   
1           1   22    0    2        1                0                 1   
2           2   49 

### 4. Chia dữ liệu thành 2 phần: Training và Testing

In [32]:
# Yêu cầu 7: Chia dữ liệu thành 80% dùng để test và 20% dùng để train.
######
# Code
X2 = df_encoded.drop(columns=['Risk'])
y2 = df_encoded['Risk']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2,random_state=42)
print(f'Train set: {X2_train.shape[0]} mẫu')
print(f'Test set : {X2_test.shape[0]} mẫu')

######

Train set: 800 mẫu
Test set : 200 mẫu


### 5. Training and Testing Naive Bayes model

In [33]:
# Yêu cầu 8: Chuẩn hóa toàn bộ dữ liệu về một phạm vi nhất định (Data Scaling)
# Gợi ý: Có thể sử dụng StandardScaler, MinMaxScaler, Robust Scaling
######

# Code
scaler = MinMaxScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)
print('X2_train_scaled shape:', X2_train_scaled.shape)
print('Giá trị min:', X2_train_scaled.min().round(3), '| Giá trị max:', X2_train_scale)
# StandardScaler (dùng cho GaussianNB)
std_scaler = StandardScaler()
X2_train_std = std_scaler.fit_transform(X2_train)
X2_test_std = std_scaler.transform(X2_test)
######

X2_train_scaled shape: (800, 10)


NameError: name 'X2_train_scale' is not defined

In [ ]:
# Training & Testing
# Yêu cầu 9: Sử dụng thư viện sklearn để xây dựng một mô hình Naive Bayes và kiểm tra kết quả phân lớp dựa trên độ đo Accuracy.
######
# Code

gnb2 = GaussianNB()
gnb2.fit(X2_train_std, y2_train)
y2_pred = gnb2.predict(X2_test_std)
acc = accuracy_score(y2_test, y2_pred)
print(f'Gaussian Naive Bayes Accuracy: {acc:.4f} ({acc*100:.2f}%)')
######

### 6. Thay đổi phân bố xác suất để chọn được bộ phân lớp phù hợp với dữ liệu

In [22]:
# Gợi ý: Khảo sát các phân bố xác suất như phân phối chuẩn (Gauss), phân phối đa thức, phân phối Bernoulli và chọn ra bộ phân lớp tương ứng với phân phối cho accuracy cao nhất.
# Mở rộng (Tùy chọn): Điều chỉnh các tham số để cho kết quả tốt hơn (VD: giá trị làm mịn alpha trong phân phối đa thức, var_smoothing trong Gauss,...)
######
# Code


######